In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1. Inicialização do PySpark
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

conf = SparkConf()
conf.set('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.11.901')
conf.set('spark.hadoop.fs.s3a.aws.credentials.provider', 'com.amazonaws.auth.InstanceProfileCredentialsProvider')

spark = SparkSession.builder.config(conf=conf).getOrCreate()

:: loading settings :: url = jar:file:/usr/local/lib/python3.7/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-32623df2-b51e-4df4-9b81-12f5f0562972;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 672ms :: artifacts dl 22ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 by [com.amazonaws#aws-java-sdk-bundle;1.12.262] in [default]
	---------------------------------------------------------------------
	|     

In [3]:

from pyspark.sql import functions as F

# Caminhos dos arquivos no S3
sellers_path = "s3a://last-mile-optimization-trusted-sofhia/dataset-orders/sellers_cleaned_dataset.csv/"
geolocation_path = "s3a://last-mile-optimization-trusted-sofhia/dataset-orders/geolocation_cleaned_dataset.csv/"

# Leitura dos datasets
sellers_df = spark.read.csv(
    sellers_path,
    header=True,
    inferSchema=True
)

geolocation_df = spark.read.csv(
    geolocation_path,
    header=True,
    inferSchema=True
)

# 1. Calcular uma coordenada representativa por CEP
geo_por_cep = (
    geolocation_df
    .filter(
        F.col("geolocation_zip_code_prefix").isNotNull() &
        F.col("geolocation_lat").isNotNull() &
        F.col("geolocation_lng").isNotNull()
    )
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        F.percentile_approx(
            "geolocation_lat", 0.5
        ).alias("geolocation_lat"),

        F.percentile_approx(
            "geolocation_lng", 0.5
        ).alias("geolocation_lng")
    )
    .withColumnRenamed(
        "geolocation_zip_code_prefix",
        "seller_zip_code_prefix"
    )
)

# 2. Associar cada vendedor à coordenada de seu CEP
joined_df = sellers_df.join(
    geo_por_cep,
    on="seller_zip_code_prefix",
    how="inner"
)

# 3. Auditoria
print("=== AUDITORIA SELLERS + GEOLOCATION ===")

print("Vendedores antes do join:", sellers_df.count())
print("Vendedores após o join:", joined_df.count())

print(
    "Vendedores sem coordenadas:",
    sellers_df.join(
        geo_por_cep,
        on="seller_zip_code_prefix",
        how="left_anti"
    ).count()
)

print(
    "CEPs duplicados na tabela agregada:",
    geo_por_cep.groupBy("seller_zip_code_prefix")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Vendedores duplicados após join:",
    joined_df.groupBy("seller_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

joined_df.show(5)
joined_df.printSchema()


26/11/03 01:07:53 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


=== AUDITORIA SELLERS + GEOLOCATION ===
Vendedores antes do join: 1849


Vendedores após o join: 1814


Vendedores sem coordenadas: 35


CEPs duplicados na tabela agregada: 0


Vendedores duplicados após join: 0


+----------------------+--------------------+-----------+-------------------+-------------------+
|seller_zip_code_prefix|           seller_id|seller_city|    geolocation_lat|    geolocation_lng|
+----------------------+--------------------+-----------+-------------------+-------------------+
|                  1021|dd55f1bb788714a40...|  SAO PAULO|-23.544184662502165| -46.63214268206478|
|                  1021|e0487761face83d64...|  SAO PAULO|-23.544184662502165| -46.63214268206478|
|                  1026|7040e82f899a04d1b...|  SAO PAULO|-23.539439503750902|-46.632843914149746|
|                  1026|630008e0f062605a4...|  SAO PAULO|-23.539439503750902|-46.632843914149746|
|                  1026|5d378b73ab7dd6f04...|  SAO PAULO|-23.539439503750902|-46.632843914149746|
+----------------------+--------------------+-----------+-------------------+-------------------+
only showing top 5 rows

root
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_id: string (nullable 

In [4]:
joined_df.coalesce(1) \
    .write \
    .option('header', 'true') \
    .mode('overwrite') \
    .csv('s3a://last-mile-optimization-trusted-sofhia/dataset-orders/join_sellers_and_geolocation.csv')

spark.stop()

26/11/03 01:08:31 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/11/03 01:08:31 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
